In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import geopandas as gpd

In [3]:
turkish_to_english_map = {
    'ç': 'c', 'ğ': 'g', 'ı': 'i', 'i̇':'i','İ': 'I', 'ö': 'o', 'ş': 's', 'ü': 'u',
    'Ç': 'C', 'Ğ': 'G', 'Ö': 'O', 'Ş': 'S', 'Ü': 'U'
}


def convert_turkish_to_english(text):
    # Replace each Turkish character in the string with its English equivalent
    if isinstance(text, str):
        return ''.join(turkish_to_english_map.get(c, c) for c in text)
    return text

In [43]:
# Import ilce_collapse
df = pd.read_excel('/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/Processed/ilce_collapse.xlsx')

df.loc[(df['Province'] == 'ADIYAMAN') & (df['District'] == 'ADIYAMAN'), 'District'] = 'merkez'
df.loc[(df['Province'] == 'KILIS') & (df['District'] == 'KILIS'), 'District'] = 'merkez'
df.loc[(df['Province'] == 'OSMANIYE') & (df['District'] == 'OSMANIYE'), 'District'] = 'merkez'

df = df.map(convert_turkish_to_english)
df['Province'] = df['Province'].str.lower()
df['District'] = df['District'].str.lower()
df['Neighborhood'] = df['Neighborhood'].apply(lambda x: x.replace('Mh.', 'Mh'))

In [44]:
provinces = (pd.unique(df["Province"]))
provinces

array(['adiyaman', 'kahramanmaras', 'malatya', 'sanliurfa', 'adana',
       'hatay', 'gaziantep', 'diyarbakir', 'osmaniye', 'kilis'],
      dtype=object)

In [47]:
# Scrape nufus data
nufus_nb = pd.DataFrame()

for provincename in provinces:
    districtname = pd.unique(df.loc[df['Province'] == provincename, 'District'])
    
    for districtname in districtname:
        url = f"https://www.nufusune.com/{districtname}-ilce-nufusu-{provincename}"
        print(url)
        
        # Send a GET request to the URL
        response = requests.get(url)
        
        # Check if the request was successful
        if response.status_code == 200:
            html_content = response.text
            print("HTML fetched successfully!")
        else:
            print(f"Failed to fetch the webpage. Status code: {response.status_code}")
            continue  # Skip to the next district if fetch fails

        # Parse the HTML using BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')

        # Extract Neighborhood-Level Data
        neighborhoods = []
        table = soup.find('div', id='ust_tablo').find('table')
        if table:
            rows = table.find('tbody').find_all('tr')
            for row in rows:
                cols = row.find_all('td')
                neighborhoods.append({
                    'Neighborhood': cols[0].text.strip(),
                    'Total Population': cols[1].text.strip(),
                    'Male Population': cols[2].text.strip(),
                    'Female Population': cols[3].text.strip()
                })
        
        # Initialize DataFrame for neighborhoods
        neighborhood_df = pd.DataFrame(neighborhoods)

        # Extract Village info if "Köy" table exists
        village_data = []
        village_table = soup.find_all('div', {'id': 'ust_tablo'})
        
        if len(village_table) > 1:  # Check if there is a second table for villages
            village_table = village_table[1].find('table')
            village_rows = village_table.find_all('tr')
            
            for row in village_rows[0:]:  # Skip the header row
                columns = row.find_all('td')
                village_data.append([col.text.strip() for col in columns])
            
            # Create a DataFrame for villages
            village_df = pd.DataFrame(village_data, columns=['Village', 'Total Population', 'Male Population', 'Female Population'])
            
            # Optionally, add a column to indicate whether the data is for a neighborhood or a village
            village_df['Type'] = 'Village'
            
            # Combine neighborhood and village DataFrames
            combined_df = pd.concat([neighborhood_df, village_df], ignore_index=True)

            # Province and dist
            combined_df['Province'] = provincename
            combined_df['District'] = districtname

            # Concatenate combined_df with the final DataFrame (nufus_nb)
            nufus_nb = pd.concat([nufus_nb, combined_df], ignore_index=True)
        
        else:
            # If no village data is found, proceed with just neighborhood data
            neighborhood_df['Type'] = 'Neighborhood'
            neighborhood_df['Province'] = provincename
            neighborhood_df['District'] = districtname
            
            nufus_nb = pd.concat([nufus_nb, neighborhood_df], ignore_index=True)


https://www.nufusune.com/merkez-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/besni-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/celikhan-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/gerger-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/golbasi-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/kahta-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/samsat-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/sincik-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/tut-ilce-nufusu-adiyaman
HTML fetched successfully!
https://www.nufusune.com/afsin-ilce-nufusu-kahramanmaras
HTML fetched successfully!
https://www.nufusune.com/andirin-ilce-nufusu-kahramanmaras
HTML fetched successfully!
https://www.nufusune.com/caglayancerit-ilce-nufusu-kahramanmaras
HTML fetched successfully!
https://www.nufusune

In [55]:
nufus = nufus_nb 

nufus

,Neighborhood,Total Population,Male Population,Female Population,Village,Type,Province,District
0,ALTINŞEHİR MAHALLESİ,26.220,12.613,13.607,NaN,NaN,adiyaman,merkez
1,YENİ MAHALLESİ,25.382,12.741,12.641,NaN,NaN,adiyaman,merkez
2,YEŞİLYURT MAHALLESİ,17.824,8.978,8.846,NaN,NaN,adiyaman,merkez
3,SİTELER MAHALLESİ,16.568,8.412,8.156,NaN,NaN,adiyaman,merkez
4,CUMHURİYET MAHALLESİ,15.883,7.943,7.940,NaN,NaN,adiyaman,merkez
...,...,...,...,...,...,...,...,...
7269,NaN,83,43,40,SÖĞÜTLÜ KÖYÜ,Village,kilis,polateli
7270,NaN,82,36,46,BEKTAŞOĞLU KÖYÜ,Village,kilis,polateli
7271,NaN,75,40,35,EĞLEN KÖYÜ,Village,kilis,polateli
7272,NaN,71,40,31,KIZILGÖL KÖYÜ,Village,kilis,polateli


In [56]:
turkish_to_english_map = {
    'ç': 'c', 'ğ': 'g', 'ı': 'i', 'i̇':'i','İ': 'I', 'ö': 'o', 'ş': 's', 'ü': 'u',
    'Ç': 'C', 'Ğ': 'G', 'Ö': 'O', 'Ş': 'S', 'Ü': 'U'
}


def convert_turkish_to_english(text):
    # Replace each Turkish character in the string with its English equivalent
    if isinstance(text, str):
        return ''.join(turkish_to_english_map.get(c, c) for c in text)
    return text

In [57]:
nufus = nufus.map(convert_turkish_to_english)

In [60]:
nufus['Neighborhood'] = nufus.apply(lambda row: row['Village'] if pd.isna(row['Neighborhood']) else row['Neighborhood'], axis=1)
nufus['Neighborhood'] = nufus['Neighborhood'].apply(lambda x: x.title().replace('Mahallesi', 'Mh'))

In [61]:
nufus = nufus[['Neighborhood', 'District', 'Province', 'Total Population']]
nufus

,Neighborhood,District,Province,Total Population
0,Altinsehir Mh,merkez,adiyaman,26.220
1,Yeni Mh,merkez,adiyaman,25.382
2,Yesilyurt Mh,merkez,adiyaman,17.824
3,Siteler Mh,merkez,adiyaman,16.568
4,Cumhuriyet Mh,merkez,adiyaman,15.883
...,...,...,...,...
7269,Sogutlu Koyu,polateli,kilis,83
7270,Bektasoglu Koyu,polateli,kilis,82
7271,Eglen Koyu,polateli,kilis,75
7272,Kizilgol Koyu,polateli,kilis,71


In [62]:
nufus.to_excel('/Users/nihankaragul/Documents/Documents - Nihan’s MacBook Air/2024 Fall/Dissertation/Data/Processed/mahalle_nufus.xlsx', index=False)